# Factorization Machines

## Load Data + Initial Preprocessing:

In [ ]:
!pip -q install torch pandas scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import sqlite3

sqlite_db_path = '/content/drive/Shareddrives/Hotels/Full_HotelRec/HotelRec.db'

# Connect to the SQLite database
conn = sqlite3.connect(sqlite_db_path)

# Get column names from the 'hotel_reviews' table
cursor = conn.cursor()
cursor.execute("PRAGMA table_info(hotel_reviews);")
columns = cursor.fetchall()
print("Columns in hotel_reviews table:")
for col in columns:
    print(col[1]) # col[1] contains the column name

#populate dataframe
ratings = pd.read_sql_query("SELECT author AS authorId, hotel_url AS hotelId, date, rating FROM hotel_reviews ORDER BY date DESC", conn, parse_dates=['date'])

conn.close()

num_authors = ratings['authorId'].nunique()
num_hotels = ratings['hotelId'].nunique()

print(f"Number of unique authors in sample: {num_authors}")
print(f"Number of unique hotels in sample: {num_hotels}")

Columns in hotel_reviews table:
hotel_url
author
date
rating
title
text
sleep_quality
value
rooms
service
cleanliness
location
Number of unique authors in sample: 21891404
Number of unique hotels in sample: 365057


In [ ]:
ratings.to_parquet('/content/drive/MyDrive/hotel_sorted.parquet')

In [ ]:
#load ratings from parquet
import pandas as pd
ratings = pd.read_parquet('/content/drive/Shareddrives/mf/hotel_sorted.parquet')

In [ ]:
len(ratings)

50264364

In [ ]:
#remove entries with missing authorIds
ratings = ratings.iloc[:50261544]

In [ ]:
#add year feature column
ratings['year'] = ratings["date"].dt.year

In [ ]:
#test-validation-train split
n = len(ratings)

test_end = int(n * 0.05)
valid_end = int(n * 0.10)

test_df = ratings.iloc[:test_end].copy()
valid_df = ratings.iloc[test_end:valid_end].copy()
train_df = ratings.iloc[valid_end:].copy()

print("Test :", len(test_df))
print("Valid:", len(valid_df))
print("Train:", len(train_df))

Test : 2513077
Valid: 2513077
Train: 45235390


##Feature Engineering

In [ ]:
#calculate hotel numeric features
hotel_stats = train_df.groupby("hotelId").agg(
    hotel_mean_rating=("rating", "mean"),
    hotel_review_count=("rating", "size")
).reset_index()

In [ ]:
#Add hotel numeric features to dataframes
train_df = train_df.merge(hotel_stats, on="hotelId", how="left")
valid_df = valid_df.merge(hotel_stats, on="hotelId", how="left")
test_df  = test_df.merge(hotel_stats, on="hotelId", how="left")

In [ ]:
#Add global mean + unknowns
global_mean = train_df["rating"].mean()

for df_ in [valid_df, test_df]:
    df_["hotel_mean_rating"] = df_["hotel_mean_rating"].fillna(global_mean)
    df_["hotel_review_count"] = df_["hotel_review_count"].fillna(0)

In [ ]:
#Sort data by date
train_df = train_df.sort_values("date", ascending=True)
valid_df = valid_df.sort_values("date", ascending=True)
test_df  = test_df.sort_values("date", ascending=True)

In [ ]:
train_df.head()

,authorId,hotelId,date,rating,year,hotel_mean_rating,hotel_review_count
45235359,CatherineMc,Hotel_Review-g187147-d233734-Reviews-Hotel_Hau...,2002-09-01,1.0,2002,2.969697,99
45235161,Discover19864,Hotel_Review-g186338-d187995-Reviews-Portobell...,2002-09-01,4.0,2002,4.161458,192
45235160,Escape17491,Hotel_Review-g644396-d153350-Reviews-Punta_Ser...,2002-09-01,5.0,2002,4.067416,89
45235159,Morgan55,Hotel_Review-g34177-d531162-Reviews-Hotel_DeFu...,2002-09-01,4.0,2002,4.686047,172
45235158,gene,Hotel_Review-g681252-d258245-Reviews-L_Auberge...,2002-09-01,1.0,2002,2.642857,14


In [ ]:
test_df.head()

,authorId,hotelId,date,rating,year,hotel_mean_rating,hotel_review_count
2513056,rockhugger48,Hotel_Review-g255111-d581997-Reviews-Ramada_Re...,2018-11-01,3.0,2018,4.016949,177.0
2321082,Culture54862329810,Hotel_Review-g155042-d181994-Reviews-SureStay_...,2018-11-01,1.0,2018,3.746622,592.0
2321083,RoyB2252,Hotel_Review-g155042-d181994-Reviews-SureStay_...,2018-11-01,5.0,2018,3.746622,592.0
2321084,HolidayBreakCustomer,Hotel_Review-g187323-d248850-Reviews-Citadines...,2018-11-01,1.0,2018,4.177419,310.0
2321085,Pioneer24133425575,Hotel_Review-g187323-d248850-Reviews-Citadines...,2018-11-01,5.0,2018,4.177419,310.0


In [ ]:
#Add unknown autor and hotel categories
train_authors = set(train_df["authorId"])
train_hotels = set(train_df["hotelId"])

def map_unknown(df):
    df["authorId"] = df["authorId"].apply(
        lambda x: x if x in train_authors else "UNK_AUTHOR"
    )
    df["hotelId"] = df["hotelId"].apply(
        lambda x: x if x in train_hotels else "UNK_HOTEL"
    )
    return df

valid_df = map_unknown(valid_df)
test_df  = map_unknown(test_df)

In [ ]:
#convert year to string
test_df["year"] = test_df["year"].astype(str)

In [ ]:
train_df["year"] = train_df["year"].astype(str)
valid_df["year"] = valid_df["year"].astype(str)

In [ ]:
from sklearn.preprocessing import LabelEncoder
import numpy as np

#encode author, hotel, and year
author_encoder = LabelEncoder()
hotel_encoder = LabelEncoder()
year_encoder = LabelEncoder()

author_values = np.append(train_df["authorId"].astype(str).unique(), "UNK_AUTHOR")
hotel_values = np.append(train_df["hotelId"].astype(str).unique(), "UNK_HOTEL")
year_values = np.append(train_df["year"].unique(), "UNK_YEAR")

author_encoder.fit(author_values)
hotel_encoder.fit(hotel_values)
year_encoder.fit(year_values)


LabelEncoder()

In [ ]:
import torch
from torch.utils.data import Dataset


In [ ]:
#add unknown category for year
train_years = set(train_df["year"])

def map_year(df):
    df["year"] = df["year"].apply(
        lambda x: x if x in train_years else "UNK_YEAR"
    )
    return df

valid_df = map_year(valid_df)
test_df  = map_year(test_df)

##Final Dataset Preparation

In [ ]:
class FMDataset(Dataset):
    def __init__(self, df):
        self.authors = torch.tensor(
            author_encoder.transform(df["authorId"]), dtype=torch.long
        )
        self.hotels = torch.tensor(
            hotel_encoder.transform(df["hotelId"]), dtype=torch.long
        )
        self.years = torch.tensor(
            year_encoder.transform(df["year"]), dtype=torch.long
        )

        self.numeric = torch.tensor(
            df[["hotel_mean_rating", "hotel_review_count"]].values,
            dtype=torch.float32
        )

        self.ratings = torch.tensor(df["rating"].values, dtype=torch.float32)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return (
            self.authors[idx],
            self.hotels[idx],
            self.years[idx],
            self.numeric[idx],
            self.ratings[idx]
        )

In [ ]:
from torch.utils.data import DataLoader

In [ ]:
train_loader = DataLoader(FMDataset(train_df), batch_size=256, shuffle=True, num_workers=2, pin_memory=True)
valid_loader = DataLoader(FMDataset(valid_df), batch_size=256, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(FMDataset(test_df), batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

##FM Model

In [ ]:
import torch
import torch.nn as nn

class HybridFM(nn.Module):
    def __init__(self, n_authors, n_hotels, n_years, n_numeric, k=20):
        super().__init__()

        # global bias
        self.global_bias = nn.Parameter(torch.zeros(1))

        # embeddings (latent factors)
        self.author_emb = nn.Embedding(n_authors, k)
        self.hotel_emb = nn.Embedding(n_hotels, k)
        self.year_emb = nn.Embedding(n_years, k)

        # biases
        self.author_bias = nn.Embedding(n_authors, 1)
        self.hotel_bias = nn.Embedding(n_hotels, 1)
        self.year_bias = nn.Embedding(n_years, 1)

        # numeric features (linear)
        self.linear_num = nn.Linear(n_numeric, 1)

        # initialize
        nn.init.normal_(self.author_emb.weight, std=0.01)
        nn.init.normal_(self.hotel_emb.weight, std=0.01)
        nn.init.normal_(self.year_emb.weight, std=0.01)

        nn.init.normal_(self.author_bias.weight, std=0.01)
        nn.init.normal_(self.hotel_bias.weight, std=0.01)
        nn.init.normal_(self.year_bias.weight, std=0.01)

    def forward(self, author, hotel, year, numeric):
        # biases
        ab = self.author_bias(author).squeeze(-1)
        hb = self.hotel_bias(hotel).squeeze(-1)
        yb = self.year_bias(year).squeeze(-1)

        # embeddings
        a = self.author_emb(author)
        h = self.hotel_emb(hotel)
        y = self.year_emb(year)

        # FM-style pairwise interactions
        interaction = (
            (a * h).sum(dim=1) +
            (a * y).sum(dim=1) +
            (h * y).sum(dim=1)
        )

        # numeric features
        num_part = self.linear_num(numeric).squeeze(-1)

        # final prediction
        return self.global_bias + ab + hb + yb + interaction + num_part

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = HybridFM(
    n_authors=len(author_encoder.classes_),
    n_hotels=len(hotel_encoder.classes_),
    n_years=len(year_encoder.classes_),
    n_numeric=2,
    k=20
).to(device)

##Defining Evaluation Metrics

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error

def evaluate_rmse(model, loader):
    model.eval()
    preds_all = []
    targets_all = []

    with torch.no_grad():
        for authors, hotels, years, numeric, ratings in loader:
            authors = authors.to(device)
            hotels = hotels.to(device)
            years = years.to(device)
            numeric = numeric.to(device)

            preds = model(authors, hotels, years, numeric)
            preds = torch.clamp(preds, 1.0, 5.0)

            preds_all.extend(preds.cpu().numpy())
            targets_all.extend(ratings.numpy())

    return np.sqrt(mean_squared_error(targets_all, preds_all))

In [ ]:
def evaluate_mae(model, loader):
    model.eval()
    preds_all = []
    targets_all = []

    with torch.no_grad():
        for authors, hotels, years, numeric, ratings in loader:
            authors = authors.to(device)
            hotels = hotels.to(device)
            years = years.to(device)
            numeric = numeric.to(device)

            preds = model(authors, hotels, years, numeric)
            preds = torch.clamp(preds, 1.0, 5.0)

            preds_all.extend(preds.cpu().numpy())
            targets_all.extend(ratings.numpy())

            # Calculate MAE
    mae = np.mean(np.abs(np.array(targets_all) - np.array(preds_all)))

    return mae

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-6)

##Training

In [ ]:
import os
import torch

os.makedirs("checkpoints", exist_ok=True)

n_epochs = 15
patience = 2
epochs_without_improvement = 0

best_valid_rmse = float("inf")
best_state = None

for epoch in range(n_epochs):
    model.train()
    total_loss = 0.0

    for authors, hotels, years, numeric, ratings in train_loader:
        authors = authors.to(device)
        hotels = hotels.to(device)
        years = years.to(device)
        numeric = numeric.to(device)
        ratings = ratings.to(device)

        optimizer.zero_grad()
        preds = model(authors, hotels, years, numeric)
        loss = criterion(preds, ratings)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(ratings)

    train_loss = total_loss / len(train_loader.dataset)
    valid_rmse = evaluate_rmse(model, valid_loader)

    print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} | Valid RMSE: {valid_rmse:.4f}")

    # save checkpoint every epoch
    checkpoint = {
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "valid_rmse": valid_rmse
    }
    torch.save(checkpoint, f"checkpoints/epoch_{epoch+1}.pth")

    # check improvement
    if valid_rmse < best_valid_rmse:
        best_valid_rmse = valid_rmse
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    # early stopping
    if epochs_without_improvement >= patience:
        print(f"Early stopping triggered at epoch {epoch+1}")
        break

Epoch 01 | Train Loss: 33.3875 | Valid RMSE: 1.1115
Epoch 02 | Train Loss: 0.9755 | Valid RMSE: 1.0538


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x79013c78ff60>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1671, in _shutdown_workers
    w.join(timeout=_utils.MP_STATUS_CHECK_INTERVAL)
  File "/usr/lib/python3.12/multiprocessing/process.py", line 149, in join
    res = self._popen.wait(timeout)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/popen_fork.py", line 40, in wait
    if not wait([self.sentinel], timeout):
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 1136, in wait
    ready = selector.select(timeout)
            ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/selectors.py", line 415, in select
    fd_event_list = self._selector.poll(timeout)
    

KeyboardInterrupt: 

##Model Evaluation

In [ ]:
valid_rmse = evaluate_rmse(model, valid_loader)

In [ ]:
#model.load_state_dict(best_state)
test_rmse = evaluate_rmse(model, test_loader)

print("Best Valid RMSE:", best_valid_rmse)
print("Test RMSE:", test_rmse)

Best Valid RMSE: 1.053788279069179
Test RMSE: 1.0970288608664256


In [ ]:
test_mae = evaluate_mae(model, test_loader)
test_mae

np.float32(0.8770994)